# RNA3D Phase 1 — RhoFold+ evaluation and 3-model comparison

**Goal:** Score the **already-completed RhoFold+ single-sequence relaxed-1000** 10-target pilot, then compare with the archived **NuFold rank-1** and **DRfold2 default** results on exactly the same frozen targets.

**CPU ONLY — no inference, checkpoint download, model installation, GPU, or GitHub modifications.** The notebook reads the existing Google Drive inputs and model outputs, clones a *pinned, read-only version* of the benchmark's evaluation helpers, builds the *same US-align release (20260527)* used previously, and saves results to a **new, immutable evaluation directory** on Google Drive.

**Scoring protocol:** `USalign <prediction_trimmed_to_observed_native_positions> <native_eval.pdb> -mol RNA -TMscore 1`. Record **TM-score normalized by Structure_2 (native)**, RMSD and alignment coverage (aligned positions / observed native residues). Compare native coordinate availability separately (observed native residues / full target length).

**Hard gates:** 10/10 input + output hashes; exact expected RhoFold+ mode (`single_sequence_quick_reference`, no MSA, `relax_steps=1000`); exact NuFold and DRfold2 legacy score calibration; full native observed-position alignment. If anything fails, stop rather than report misleading figures.

⚠️ **Scientific scope:** This is an exploratory comparison. The RhoFold+ single-sequence mode is described upstream as quick-reference/ablation, *not* full MSA-based RhoFold+. The models also differ in architecture, preprocessing, and candidate selection. Several native structures have missing residues. No template-sparsity conclusion or definitive model ranking should be inferred from 10 pilot cases.

**Run cells top to bottom.** If a check fails, send the full error output for diagnosis. Do not delete or overwrite any existing run or evaluation folder.

## 1. Mount the Drive account with `RNA3D_Benchmark`

Select **Runtime → Change runtime type → CPU / None** in Colab. GPU access and compute units are **not** required for this evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Freeze paths, model identities and output ID

In [ ]:
from pathlib import Path
import csv, json, math, os, re, shutil, subprocess, sys
from datetime import datetime, timezone
from statistics import mean, median

ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
NUFOLD_RUN=ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
DRFOLD2_RUN=ROOT/'DRfold2/runs/drfold2_pilot10_20261110_v1'
RHO_RUN=ROOT/'RhoFold/runs/rhofoldplus_pilot10_single_relax1000_20261111_v1'
NATIVE=ROOT/'native/pilot_v1/native_eval'
PREV_EVAL=DRFOLD2_RUN/'evaluations/drfold2_vs_nufold_usalign20260527_20261111_v1'

TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
         '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']

EXPECTED_RHO_COMMIT='6bdfbda720184409eb682ce08c05d258162ddc48'
EXPECTED_DR_COMMIT='3667d6ac44c5b14cff94c36e0371224acb2cdae5'
PROJECT_URL='https://github.com/ABDULSALAM589/rna3d-template-sparse-benchmark.git'
PROJECT_COMMIT='c79a78b0a35be00a31aadeb04a6ebec5f2a0004f'
PROJECT=Path('/content/rna3d-rhofoldplus-eval-code')
USALIGN_TAG='20260527'
USALIGN_SRC=Path('/content/USalign-rho-eval')
USALIGN=USALIGN_SRC/'USalign'

# This ID is intentionally new and immutable. If it already exists, stop and inspect instead of overwriting.
EVALUATION_ID='rhofoldplus_threeway_usalign20260527_20261112_v1'
EVALUATION_DIR=RHO_RUN/'evaluations'/EVALUATION_ID
STAGE=Path('/content/rna3d_rhofoldplus_eval_stage_20261112')

assert len(TARGETS)==len(set(TARGETS))==10
for p in [ROOT,NUFOLD_RUN,DRFOLD2_RUN,RHO_RUN,NATIVE,PREV_EVAL]:
    assert p.is_dir(), f'Missing Drive folder: {p}'
print('NuFold run:',NUFOLD_RUN)
print('DRfold2 run:',DRFOLD2_RUN)
print('RhoFold+ run:',RHO_RUN)
print('Native references:',NATIVE)
print('Output evaluation folder:',EVALUATION_DIR)
print('No GPU or inference needed.')


## 3. Load read-only, previously used residue-mapping code

Pin the project commit to avoid silently changing the algorithm after the previous evaluation. The clone is local to temporary `/content` and **does not modify or push** to GitHub.

In [ ]:
def run_checked(cmd,cwd=None):
    cmd=[str(a) for a in cmd]
    print('$',' '.join(cmd))
    subprocess.run(cmd,cwd=cwd,check=True)

if not PROJECT.exists():
    run_checked(['git','clone','--filter=blob:none',PROJECT_URL,PROJECT])
run_checked(['git','-C',PROJECT,'checkout','--detach',PROJECT_COMMIT])
actual_commit=subprocess.check_output(['git','-C',PROJECT,'rev-parse','HEAD'],text=True).strip()
assert actual_commit==PROJECT_COMMIT
sys.path.insert(0,str(PROJECT/'scripts/nufold'))
from common import sha256_file, pdb_residues, validate_ranked_pdb, read_native_positions, write_mapped_prediction
print('Pinned mapping helpers verified:',actual_commit)


## 4. Build pinned US-align on CPU

This is the same release that reproduced all 10 archived NuFold scores in the earlier evaluation. Compiling US-align is lightweight relative to RNA model inference.

In [ ]:
if not USALIGN_SRC.exists():
    run_checked(['git','clone','--depth','1','--branch',USALIGN_TAG,
                 'https://github.com/pylelab/USalign.git',USALIGN_SRC])
tag_head=subprocess.check_output(['git','-C',USALIGN_SRC,'rev-parse','HEAD'],text=True).strip()
tag_commit=subprocess.check_output(['git','-C',USALIGN_SRC,'rev-parse',f'refs/tags/{USALIGN_TAG}^{{commit}}'],text=True).strip()
assert tag_head==tag_commit, 'US-align source does not match pinned tag'
if not USALIGN.is_file():
    run_checked(['make'],cwd=USALIGN_SRC)
assert USALIGN.is_file() and os.access(USALIGN,os.X_OK)
USALIGN_SHA=sha256_file(USALIGN)
print('US-align pinned source commit:',tag_head)
print('US-align executable SHA256:',USALIGN_SHA)


## 5. Verify the 10 RhoFold+ predictions, FASTA, reference maps and previous evaluations

No PDB file will be edited. The observed native positions are used **only for evaluation**, never for inference.

In [ ]:
def read_csv(path):
    assert path.is_file(),f'Missing CSV: {path}'
    with path.open(newline='') as f:
        return list(csv.DictReader(f))

def fasta_sequence(path):
    lines=[x.strip() for x in path.read_text().splitlines() if x.strip()]
    assert lines and lines[0].startswith('>') and sum(x.startswith('>') for x in lines)==1, path
    seq=''.join(x for x in lines if not x.startswith('>')).upper()
    assert seq and set(seq)<=set('ACGU'),path
    return seq

nu_rows=read_csv(NUFOLD_RUN/'analysis/rank1_results.csv')
nu_scores={r['target_id']:r for r in nu_rows if r.get('rank')=='1' and r.get('status')=='scored'}
dr_rows=read_csv(PREV_EVAL/'drfold2_scores.csv')
dr_scores={r['target_id']:r for r in dr_rows if r.get('status')=='scored'}
assert set(nu_scores)==set(dr_scores)==set(TARGETS), 'Missing/duplicate NuFold or DRfold2 scores'

# Trust the prior DRfold2 evaluation only after confirming its immutable output hashes.
prev_final=PREV_EVAL/'FINALIZED.json'
assert prev_final.is_file(),f'Previous DRfold2 evaluation not finalized: {prev_final}'
prev_marker=json.loads(prev_final.read_text())
assert prev_marker['status']=='complete'
for name,digest in prev_marker['files'].items():
    assert sha256_file(PREV_EVAL/name)==digest,f'Prior evaluation file changed: {name}'

inputs=[]
checkpoint_hashes=set()
for t in TARGETS:
    fas=NUFOLD_RUN/'inputs'/t/f'{t}.fasta'
    native=NATIVE/f'{t}_native_eval.pdb'
    rmap=NATIVE/f'{t}_residue_map.csv'
    folder=RHO_RUN/t
    marker=folder/'complete.json'
    assert all(p.is_file() for p in [fas,native,rmap,marker]),f'{t}: missing input/native/complete marker'
    seq=fasta_sequence(fas)
    meta=json.loads(marker.read_text())
    pred=folder/meta.get('primary_pdb_name','MISSING')
    assert pred.is_file(),f'{t}: missing prediction PDB: {pred}'
    assert meta.get('status')=='complete' and meta.get('target_id')==t
    assert meta.get('model')=='RhoFold+' and meta.get('model_commit')==EXPECTED_RHO_COMMIT
    assert meta.get('input_mode')=='single_sequence_quick_reference'
    assert meta.get('single_seq_pred') is True and meta.get('msa_depth')==1
    assert meta.get('input_msa') is None
    assert meta.get('relax_steps')==1000
    assert meta.get('primary_pdb_name')=='relaxed_1000_model.pdb',f'{t}: unexpected primary PDB selection'
    assert meta.get('input_length')==len(seq)
    assert sha256_file(fas)==meta['fasta_sha256'],f'{t}: FASTA hash mismatch'
    assert sha256_file(pred)==meta['primary_pdb_sha256'],f'{t}: PDB hash mismatch'
    ckpt_sha=meta.get('checkpoint_sha256')
    assert ckpt_sha and re.fullmatch(r'[a-f0-9]{64}',ckpt_sha),f'{t}: invalid checkpoint SHA'
    checkpoint_hashes.add(ckpt_sha)
    pred_info=validate_ranked_pdb(pred,len(seq))
    assert pred_info['sequence']==seq,f'{t}: RhoFold+ PDB sequence mismatch'
    native_positions=read_native_positions(rmap)
    assert native_positions and native_positions==sorted(set(native_positions))
    assert 1<=native_positions[0] and native_positions[-1]<=len(seq)
    native_res=pdb_residues(native)
    assert len(native_res)==len(native_positions),f'{t}: native PDB vs map count mismatch'
    assert ''.join(r['one'] for r in native_res)==''.join(seq[i-1] for i in native_positions),f'{t}: native sequence vs map mismatch'
    # Prior DRfold2 run used exactly the same FASTA and experimental references.
    dr_meta=json.loads((DRFOLD2_RUN/t/'complete.json').read_text())
    assert dr_meta['model_commit']==EXPECTED_DR_COMMIT
    assert dr_meta['fasta_sha256']==meta['fasta_sha256']
    assert dr_meta['native_sha256']==sha256_file(native)
    assert dr_meta['pdb_sha256']==dr_scores[t]['prediction_sha256']
    assert int(dr_scores[t]['native_residues'])==len(native_positions)
    assert int(dr_scores[t]['input_length'])==len(seq)
    assert int(nu_scores[t]['native_residues'])==len(native_positions)
    inputs.append({'target_id':t,'sequence':seq,'fasta':fas,'native':native,'map':rmap,
                   'pred':pred,'meta':meta,'positions':native_positions,
                   'length':len(seq),'native_count':len(native_positions),
                   'native_fraction':len(native_positions)/len(seq)})
assert len(checkpoint_hashes)==1,'Mixed RhoFold+ checkpoints: stop and investigate'
RHO_CKPT_SHA=next(iter(checkpoint_hashes))
assert len(inputs)==10
print('PRECHECK PASS: 10/10 RhoFold+ predictions + frozen FASTAs + native mapping + previous DRfold2 evaluation hashes')
print('RhoFold+ checkpoint SHA256:',RHO_CKPT_SHA)
for q in inputs:
    print(f"{q['target_id']:9s} : {q['length']:2d} nt, native observed {q['native_count']:2d}/{q['length']:2d} ({q['native_fraction']:.1%})")


## 6. Shared RNA-specific US-align parser + test

Insist on native-normalized TM-score (`Structure_2`), not the first TM-score line. This parser supports the archived US-align v20260527 text format.

In [ ]:
def parse_scoring(stdout,native_count):
    aligned=re.search(r'Aligned length\s*=\s*(\d+)\s*,\s*RMSD\s*=\s*([\d.eE+-]+)',stdout,re.I)
    if aligned:
        common=int(aligned.group(1));rmsd=float(aligned.group(2))
    else:
        cm=re.search(r'Number of residues in common\s*=\s*(\d+)',stdout,re.I)
        rm=re.search(r'RMSD of the common residues\s*=\s*([\d.eE+-]+)',stdout,re.I)
        if not(cm and rm):
            raise ValueError('Missing aligned length and RMSD')
        common=int(cm.group(1));rmsd=float(rm.group(1))
    rms=re.findall(r'TM-score\s*=\s*([\d.eE+-]+)\s*\(normalized by length of Structure_2\b[^)]*\)',stdout,re.I)
    if not rms:
        raise ValueError('Native-normalized Structure_2 TM-score missing')
    tm=float(rms[-1])
    assert 0<common<=native_count and math.isfinite(rmsd) and rmsd>=0
    assert math.isfinite(tm) and 0<=tm<=1.00001
    return {'alignment_residues':common,'alignment_coverage':common/native_count,
            'tm_score':tm,'rmsd_angstrom':rmsd}

def score_mapped(mapped,native):
    cmd=[str(USALIGN),str(mapped),str(native),'-mol','RNA','-TMscore','1']
    p=subprocess.run(cmd,capture_output=True,text=True)
    log=p.stdout+('\n--- STDERR ---\n'+p.stderr if p.stderr else '')
    if p.returncode:
        raise RuntimeError(f'US-align failed code {p.returncode}: {log[-900:]}')
    return log

sample="""Aligned length= 13, RMSD=  11.52, Seq_ID=n_identical/n_aligned= 1.000
TM-score= 0.24331 (normalized by length of Structure_1: L=13, d0=0.40)
TM-score= 0.24331 (normalized by length of Structure_2: L=13, d0=0.40)"""
assert parse_scoring(sample,13)=={'alignment_residues':13,'alignment_coverage':1.0,
                                  'tm_score':0.24331,'rmsd_angstrom':11.52}
print('US-align parser test: PASS')


## 7. Calibrate against **both** previously scored models (10/10 each)

Re-score the already mapped NuFold and DRfold2 PDBs using the freshly built US-align binary. Stop if the scores differ beyond precision of archived printed scores. This is a **scoring-environment sanity test** and does not rerun the prediction models.

In [ ]:
nu_calibration=[]
dr_calibration=[]
for q in inputs:
    t=q['target_id']
    nu_map=NUFOLD_RUN/'evaluations/pilot10_native_eval_v2'/t/f'{t}_rank_1_mapped.pdb'
    dr_map=PREV_EVAL/'mapped'/f'{t}_mapped.pdb'
    assert nu_map.is_file() and dr_map.is_file(),f'{t}: calibration mapped PDB missing'
    for mapped,expected,collector,label in [(nu_map,nu_scores[t],nu_calibration,'NuFold'),
                                             (dr_map,dr_scores[t],dr_calibration,'DRfold2')]:
        actual=parse_scoring(score_mapped(mapped,q['native']),q['native_count'])
        dt=abs(actual['tm_score']-float(expected['tm_score']))
        dr=abs(actual['rmsd_angstrom']-float(expected['rmsd_angstrom']))
        assert dt<=0.00006 and dr<=0.06,(t,label,'score drift',actual,expected,dt,dr)
        assert actual['alignment_residues']==int(expected['alignment_residues']), (t,label,'coverage drift')
        collector.append({'target_id':t,'model':label,'tm_abs_diff':dt,'rmsd_abs_diff_angstrom':dr})
assert len(nu_calibration)==len(dr_calibration)==10
print('NUFOLD GOLDEN CALIBRATION PASS: 10/10')
print('DRFOLD2 GOLDEN CALIBRATION PASS: 10/10')
print('Max NuFold/DRfold2 TM discrepancy:',max(r['tm_abs_diff'] for r in nu_calibration+dr_calibration))


## 8. Score **one RhoFold+ target** as a dry-run (CPU)

Creates temporary mapped output under `/content` only. This first run confirms PDB residue handling and US-align formatting. Nothing has been saved to Drive yet.

In [ ]:
assert len(nu_calibration)==len(dr_calibration)==10
if STAGE.exists():
    raise FileExistsError(f'Local staging exists: {STAGE}. Inspect it before rerunning.')
STAGE.mkdir(parents=True)
(STAGE/'mapped').mkdir()
(STAGE/'logs').mkdir()

first=inputs[0]
t=first['target_id']
mapped=STAGE/'mapped'/f'{t}_mapped.pdb'
kept=write_mapped_prediction(first['pred'],mapped,first['sequence'],first['positions'])
assert kept==first['native_count']
out=score_mapped(mapped,first['native'])
(STAGE/'logs'/f'{t}_usalign.txt').write_text(out)
sc=parse_scoring(out,first['native_count'])
assert sc['alignment_residues']==first['native_count'],f'{t}: incomplete mapped alignment'
first_score=sc
print(f"RhoFold+ PILOT SCORING PASS: {t}, TM={sc['tm_score']:.5f}, RMSD={sc['rmsd_angstrom']:.2f} Å, coverage={sc['alignment_coverage']:.1%}")


## 9. Score all 10 RhoFold+ predictions — local staging only

The pilot result is reused; each other prediction is trimmed to the *same* experimental native positions. Any alignment failure stops the notebook and prevents the final Drive publish cell.

In [ ]:
assert len(nu_calibration)==len(dr_calibration)==10
results=[]
for q in inputs:
    t=q['target_id']
    if t==inputs[0]['target_id']:
        s=first_score
    else:
        mapped=STAGE/'mapped'/f'{t}_mapped.pdb'
        kept=write_mapped_prediction(q['pred'],mapped,q['sequence'],q['positions'])
        assert kept==q['native_count'],f'{t}: map position count mismatch'
        output=score_mapped(mapped,q['native'])
        (STAGE/'logs'/f'{t}_usalign.txt').write_text(output)
        s=parse_scoring(output,q['native_count'])
    assert s['alignment_residues']==q['native_count'],f'{t}: not all observed native residues aligned'
    results.append({'target_id':t,'status':'scored',
        'input_length':q['length'],'native_residues':q['native_count'],
        'native_observed_fraction':q['native_fraction'],
        'alignment_residues':s['alignment_residues'],
        'alignment_coverage':s['alignment_coverage'],
        'tm_score':s['tm_score'],'rmsd_angstrom':s['rmsd_angstrom'],
        'inference_runtime_seconds':q['meta']['seconds'],
        'primary_pdb_name':q['meta']['primary_pdb_name'],
        'prediction_sha256':q['meta']['primary_pdb_sha256'],
        'checkpoint_sha256':q['meta']['checkpoint_sha256'],
        'model_commit':q['meta']['model_commit'],
        'input_mode':q['meta']['input_mode'],
        'relax_steps':q['meta']['relax_steps']})
    print(f"{t:9s}: RhoFold+ TM {s['tm_score']:.5f}, RMSD {s['rmsd_angstrom']:.2f} Å | native {q['native_fraction']:.0%}")
assert len(results)==10 and all(r['status']=='scored' for r in results)
print('RHOFOLD+ STRUCTURAL SCORING PASS: 10/10 (staged locally; NOT yet saved to Drive)')


## 10. NuFold × DRfold2 × RhoFold+ paired comparison

Main scores use identical native reference PDBs and aligned residue maps. Aggregate on **all 10** and separately on **native-coordinate availability ≥80%** (8 targets). Very small score differences are not meaningful evidence of one model's superiority. RMSD is not directly equivalent to TM-score, and the sample is too small for template-sparsity claims.

In [ ]:
paired=[]
for r in results:
    t=r['target_id']; nu=nu_scores[t]; dr=dr_scores[t]
    a=float(nu['tm_score']); b=float(dr['tm_score']); c=r['tm_score']
    ar=float(nu['rmsd_angstrom']);br=float(dr['rmsd_angstrom']);cr=r['rmsd_angstrom']
    assert int(nu['native_residues'])==int(dr['native_residues'])==r['native_residues']
    paired.append({'target_id':t,'input_length':r['input_length'],
        'native_residues':r['native_residues'],'native_observed_fraction':r['native_observed_fraction'],
        'nufold_rank1_tm_score':a,'drfold2_tm_score':b,'rhofoldplus_single_tm_score':c,
        'delta_tm_rho_minus_nufold':c-a,'delta_tm_rho_minus_drfold2':c-b,
        'nufold_rank1_rmsd_angstrom':ar,'drfold2_rmsd_angstrom':br,
        'rhofoldplus_single_rmsd_angstrom':cr,
        'native_alignment_coverage':r['alignment_coverage']})

MODEL_KEYS={'NuFold':'nufold_rank1','DRfold2':'drfold2','RhoFold+ (single)':'rhofoldplus_single'}
def aggregate(rows):
    if not rows: return {'n':0}
    data={'n':len(rows)}
    for name,key in MODEL_KEYS.items():
        tm=[r[f'{key}_tm_score'] for r in rows]
        rmsd=[r[f'{key}_rmsd_angstrom'] for r in rows]
        data[key]={'mean_tm_score':mean(tm),'median_tm_score':median(tm),
                   'mean_rmsd_angstrom':mean(rmsd),'median_rmsd_angstrom':median(rmsd)}
    return data

high=[r for r in paired if r['native_observed_fraction']>=0.80]
summary={'all_ten_exploratory':aggregate(paired),
         'native_observed_fraction_ge_0p80':aggregate(high),
         'caveat':'Exploratory, 10 targets only; RhoFold+ single-sequence mode is not its full MSA method; candidate selection differs; incomplete native coordinates affect assessment.'}
print('THREE-MODEL COMPARISON READY:',len(paired),'/ 10')
print('HIGH NATIVE-COVERAGE TARGETS:',len(high),'of 10')
for subset_name,sub in [('all 10',summary['all_ten_exploratory']),('native observed >=80%',summary['native_observed_fraction_ge_0p80'])]:
    print('\n',subset_name, f"(n={sub['n']})")
    for name,key in MODEL_KEYS.items():
        s=sub[key]
        print(f"  {name:20s} mean TM={s['mean_tm_score']:.5f}; mean RMSD={s['mean_rmsd_angstrom']:.2f} Å")
print('\nPer-target RhoFold+ comparison:')
for r in paired:
    print(f"{r['target_id']:9s} NuFold={r['nufold_rank1_tm_score']:.5f} DRfold2={r['drfold2_tm_score']:.5f} RhoFold+={r['rhofoldplus_single_tm_score']:.5f} native={r['native_observed_fraction']:.0%}")


## 11. Save to a **new, immutable** Google Drive evaluation directory

Writes `rhofoldplus_scores.csv`, `three_model_paired.csv`, both calibration CSVs, `summary.json`, and local mapped structures and alignment logs. After copying and checking file hashes, writes a `FINALIZED.json` marker **last**. If the output directory already exists, choose a new `EVALUATION_ID` rather than overwriting it. Original model predictions remain unchanged.

In [ ]:
def write_csv(path, rows):
    assert rows
    with path.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0].keys()),lineterminator='\n')
        writer.writeheader(); writer.writerows(rows)

assert len(results)==len(paired)==len(nu_calibration)==len(dr_calibration)==10
write_csv(STAGE/'rhofoldplus_scores.csv',results)
write_csv(STAGE/'three_model_paired.csv',paired)
write_csv(STAGE/'nufold_calibration.csv',nu_calibration)
write_csv(STAGE/'drfold2_calibration.csv',dr_calibration)

meta={'evaluation_id':EVALUATION_ID,
      'created_utc':datetime.now(timezone.utc).isoformat(),
      'source_runs':{
          'nufold':'pilot10_single_20261018T203832Z',
          'drfold2':'drfold2_pilot10_20261110_v1',
          'rhofoldplus':'rhofoldplus_pilot10_single_relax1000_20261111_v1'},
      'source_drfold2_evaluation':'drfold2_vs_nufold_usalign20260527_20261111_v1',
      'pinned_mapping_project_commit':actual_commit,
      'usalign_tag':USALIGN_TAG,'usalign_source_commit':tag_head,
      'usalign_binary_sha256':USALIGN_SHA,
      'usalign_command':'USalign <mapped_prediction> <native> -mol RNA -TMscore 1',
      'tm_score_normalization':'Structure_2 (experimental native)',
      'reference_policy':'Observed native target positions only',
      'model_details':{
          'nufold':'NuFold rank-1, single-record A3M + predicted IPknot SS',
          'drfold2':'DRfold2 upstream default single-output sequence-only FASTA',
          'rhofoldplus':'RhoFold+ single_sequence_quick_reference, MSA depth 1, relaxed_1000_model.pdb',
          'rhofoldplus_commit':EXPECTED_RHO_COMMIT,
          'rhofoldplus_checkpoint_sha256':RHO_CKPT_SHA,
          'drfold2_commit':EXPECTED_DR_COMMIT},
      'counts':{'targets_scored':len(results),'nufold_calibration_pass':len(nu_calibration),
                'drfold2_calibration_pass':len(dr_calibration)},
      'summary':summary,
      'limitations':['Pilot results only. Model inference was not repeated.',
          'RhoFold+ single-sequence is not its full MSA benchmark condition.',
          'Training-data overlap and structural-template scarcity require separate analysis.',
          'Models have different input and output-selection policies.',
          'Some native structures lack coordinates for substantial sequence fractions.',
          'Native-alignment coverage 100% does not imply 100% native coordinate availability.']}
(STAGE/'summary.json').write_text(json.dumps(meta,indent=2)+'\n')

assert not EVALUATION_DIR.exists(),f'Output exists: {EVALUATION_DIR}. DO NOT overwrite; inspect or choose a new EVALUATION_ID.'
EVALUATION_DIR.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,EVALUATION_DIR)
required=['rhofoldplus_scores.csv','three_model_paired.csv',
          'nufold_calibration.csv','drfold2_calibration.csv','summary.json']
for f in required:
    assert sha256_file(EVALUATION_DIR/f)==sha256_file(STAGE/f),f'Copy mismatch: {f}'
(EVALUATION_DIR/'FINALIZED.json').write_text(json.dumps({
    'status':'complete','evaluation_id':EVALUATION_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'files':{f:sha256_file(EVALUATION_DIR/f) for f in required}},indent=2)+'\n')
print('SAVED, VERIFIED:',EVALUATION_DIR)
print('THREE-MODEL CSV:',EVALUATION_DIR/'three_model_paired.csv')
print('No GitHub commit or push was performed.')


## 12. Final **read-only** verification (can be rerun after Colab reconnects)

On a later CPU or non-GPU session, run the Drive mount + settings cells, then this verification cell. You do **not** need to rebuild US-align or load model checkpoints to inspect saved evaluation results.

In [ ]:
final_path=EVALUATION_DIR/'FINALIZED.json'
assert final_path.is_file(), f'Output is not finalized: {final_path}'
final=json.loads(final_path.read_text())
assert final.get('status')=='complete' and final.get('evaluation_id')==EVALUATION_ID
for name,digest in final['files'].items():
    assert sha256_file(EVALUATION_DIR/name)==digest,f'Checksum mismatch: {name}'
print('EVALUATION ARTIFACT VERIFICATION PASS')
print('Files:',', '.join(sorted(p.name for p in EVALUATION_DIR.iterdir())))
print('Summary:',EVALUATION_DIR/'summary.json')
print('Comparison:',EVALUATION_DIR/'three_model_paired.csv')
